In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
def get_json(url):
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    return data

In [0]:
def fetch_all(first_url):
    list_of_records = []

    url = first_url

    while url:
        data = get_json(url)

        if 'data' in data:
            list_of_records.extend(data['data'])
        
        next_page = data.get('next')
        if next_page:
            url = urljoin(url, next_page)
        else:
            url = None
    
    return list_of_records

In [0]:
def save_json(records, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(records, f,  ensure_ascii=False, indent=2)

In [0]:
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here
    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes, unchanged
    
    return len(response.content)

In [0]:
def extract(raw_dir):
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))

    download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))

    orders_df = pd.json_normalize(orders, sep='_')

    orders_df = orders_df.rename(columns={'customer_name': 'customer', 'customer_city': 'city'})

    products_df = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    teams_df = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    matches_df = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))

    return {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df
        }

In [0]:
def clean_products(df):
    cleaned = df.drop_duplicates(subset='product_id').copy()

    cleaned['team_code'] = cleaned['team_code'].fillna('').str.strip().str.upper()
    cleaned['category'] = cleaned['category'].str.strip().str.lower()
    
    cleaned['price'] = cleaned['price'].str.replace(' RSD', '', regex=False)
    cleaned['price'] = cleaned['price'].str.replace(',', '.', regex=False)
    cleaned['price'] = pd.to_numeric(cleaned['price'])

    cleaned['active'] = cleaned['active'].str.strip().str.upper().isin(['YES'])

    return cleaned

In [0]:
def clean_orders(df):
    cleaned = df.copy()

    cleaned['order_id'] = cleaned['order_id'].str.strip().str.upper()
    cleaned['customer'] = cleaned['customer'].str.strip().str.title()
    cleaned['city'] = cleaned['city'].str.strip().str.title().fillna('Unknown')
    cleaned['payment'] = cleaned['payment'].str.strip()

    return cleaned

In [0]:
def remove_duplicate_orders(df):
    unique = df.drop_duplicates(subset='order_id', keep='first')
    duplicates = df[df.duplicated(subset='order_id', keep='first')]

    return unique, duplicates

In [0]:
def validate_order(row, product_ids):
    order_id = str(row.get('order_id'))

    if not order_id.startswith('NSS-'):
        return 'invalid_order'

    raw_quantity = row.get('quantity')
    if pd.isna(raw_quantity):
        return 'missing_quantity'

    quantity = pd.to_numeric(raw_quantity, errors='coerce')
    if pd.isna(quantity) or quantity <= 0 or quantity % 1 != 0:
        return 'invalid_quantity'

    price = pd.to_numeric(row.get('unit_price'), errors='coerce')
    if pd.isna(price) or price <= 0:
        return 'invalid_price'

    if row.get('product_id') not in product_ids:
        return 'unknown_product'

    return 'OK'

In [0]:
def transform_orders(df, products):
    transformed = df.copy()

    transformed['quantity'] = pd.to_numeric(transformed['quantity'], errors='coerce').astype(int)
    transformed['unit_price'] = pd.to_numeric(transformed['unit_price'], errors='coerce').astype(float)
    transformed['order_date'] = pd.to_datetime(transformed['order_date'])
    transformed['total'] = transformed['quantity'] * transformed['unit_price']

    transformed = transformed.merge(
        products[['product_id', 'product_name', 'team_code', 'category']],
        on='product_id',
        how='left'
        )
    
    return transformed

In [ ]:
def split_orders(df, products):
    cleaned = clean_orders(df)
    unique, duplicates = remove_duplicate_orders(cleaned)
    product_ids = set(products['product_id'])

    reasons = unique.apply(lambda row: validate_order(row.to_dict(), product_ids), axis=1)
    valid = transform_orders(unique[reasons == 'OK'], products).reset_index(drop=True)
    invalid = unique[reasons != 'OK'].assign(reason=reasons[reasons != 'OK'])
    dup_with_reason = duplicates.assign(reason='duplicate')

    rejected = pd.concat([invalid, dup_with_reason]).reset_index(drop=True)

    return valid, rejected

In [ ]:
def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    played = matches[matches['status'] == 'finished'].copy()
    kickoff = pd.to_datetime(played['kickoff_ts'], errors='coerce').dt.normalize()
    codes = dict(zip(teams['team_id'], teams['code']))

    home = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': kickoff,
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals'],
    })

    away = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': kickoff,
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals'],
    })

    combined = pd.concat([home, away], ignore_index=True)

    combined['result'] = 'L'
    combined.loc[combined['goals_for'] == combined['goals_against'], 'result'] = 'D'
    combined.loc[combined['goals_for'] > combined['goals_against'], 'result'] = 'W'
    combined['points'] = combined['result'].map({'W': 3, 'D': 1, 'L': 0})

    return combined

In [ ]:
def league_table(results):
    grouped = results.groupby('team_code', as_index=False).agg(
        played=('match_id', 'count'),
        won=('result', lambda s: (s == 'W').sum()),
        drawn=('result', lambda s: (s == 'D').sum()),
        lost=('result', lambda s: (s == 'L').sum()),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
        points=('points', 'sum'),
    )

    grouped['goal_diff'] = grouped['goals_for'] - grouped['goals_against']

    table = grouped.sort_values(by=['points', 'goal_diff', 'goals_for'], ascending=[False, False, False]).reset_index(drop=True)

    table['position'] = table.index + 1

    columns = [
        'position',
        'team_code',
        'played',
        'won',
        'drawn',
        'lost',
        'goals_for',
        'goals_against',
        'goal_diff',
        'points',
    ]
    
    return table[columns]

In [ ]:
def match_windows(results):
    parts = []

    for d in range(4):
        part = results[['team_code', 'match_date', 'result']].copy()
        part['date'] = pd.to_datetime(part['match_date']) + pd.to_timedelta(d, unit='D')
        parts.append(part)

    combined = pd.concat(parts, ignore_index=True)
    out = combined[['team_code', 'date', 'result']].rename(columns={'result': 'window'})
    
    return out

In [ ]:
def revenue_by(df, column):
    out = (
        df.groupby(column, as_index=False)['total']
        .sum()
        .sort_values('total', ascending=False)
        .reset_index(drop=True)
    )
    out = out[[column, 'total']]
    
    return out

In [ ]:
def sales_after_results(valid, windows):
    valid_df = valid.copy()
    win_df = windows.copy()

    valid_df['order_date'] = pd.to_datetime(
        valid_df['order_date'], errors='coerce').dt.normalize()
    win_df['date'] = pd.to_datetime(win_df['date'], errors='coerce').dt.normalize()

    teams = pd.DataFrame({'team_code': win_df['team_code'].dropna().unique()})
    days = pd.DataFrame({'order_date': valid_df['order_date'].dropna().unique()})
    calendar = teams.merge(days, how='cross')

    daily_rev = (
        valid_df[valid_df['team_code'].notna() & (valid_df['team_code'] != '')]
        .groupby(['team_code', 'order_date'], as_index=False)['total']
        .sum()
        .rename(columns={'total': 'revenue'})
    )

    calendar = calendar.merge(daily_rev, on=['team_code', 'order_date'], how='left')
    calendar['revenue'] = calendar['revenue'].fillna(0.0)

    calendar = calendar.merge(
        win_df[['team_code', 'date', 'window']],
        left_on=['team_code', 'order_date'],
        right_on=['team_code', 'date'],
        how='left',
    )

    calendar['window'] = calendar['window'].fillna('none')

    out = (
        calendar.groupby('window', as_index=False).agg(
            team_days=('order_date', 'count'),
            revenue=('revenue', 'sum'),
        )
        .sort_values('revenue', ascending=False)
        .reset_index(drop=True)
    )

    out['avg_daily_revenue'] = (out['revenue'] / out['team_days']).round(2)
    out = out[['window', 'team_days', 'revenue', 'avg_daily_revenue']]

    return out

In [0]:
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}

    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)

        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
            
        written[name] = len(df)

    return written

In [0]:
def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(valid.dropna(subset=
        ['team_code']), 'team_code'),
        'revenue_by_category.csv': revenue_by(valid, 'category'),
        'sales_after_results.csv': sales_after_results(valid, windows),
    }

    written = load(outputs, processed_dir)

    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int((rejected['reason'] == 'duplicate').sum()),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(float(valid['total'].sum()), 2),
        'files_written': len(written) + 1,
    }

    pd.DataFrame(list(summary.items()), columns=['metric', 'value']).to_csv(os.path.join(processed_dir, 'summary.csv'), index=False)

    return summary

In [ ]:
run_pipeline(RAW_DIR, PROCESSED_DIR)